# Logoloc: treino da ResNet-50 com P1 e P2

Treina o Faster R-CNN com ResNet-50 sobre P1 e P2 juntos, 40 imagens por classe, com a mesma receita do treino original. O run se chama `frcnn_r50_p1p2_v1`. Depois do treino, rode o `logoloc_colab_eval.ipynb` para os experimentos.

**Antes de rodar**: `Ambiente de execução -> Alterar tipo de ambiente de execução -> GPU`.

O treino leva várias horas. **Se a sessão cair**, use `Ambiente de execução -> Executar tudo` de novo: o treino retoma do último checkpoint salvo na Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import subprocess
import zipfile

DRIVE_DIR = '/content/drive/MyDrive/TCC II'
assert os.path.exists(DRIVE_DIR), f'Pasta nao encontrada: {DRIVE_DIR}'

RUN_NAME = 'frcnn_r50_p1p2_v1'
BASE_CONFIG = 'configs/base_p1p2.yaml'
MODEL_CONFIG = 'configs/faster_rcnn/resnet50.yaml'

print('OK', os.listdir(DRIVE_DIR))

## 1. Código

In [ ]:
PROJECT_DIR = '/content/tcc'
REPO = 'https://github.com/lucaskluuug/tcc.git'

os.chdir('/content')
if os.path.isdir(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
!git clone -q $REPO $PROJECT_DIR

%cd $PROJECT_DIR
assert os.path.exists(BASE_CONFIG), f'{BASE_CONFIG} nao existe: o codigo clonado esta desatualizado'
!git log --oneline -1
print('OK')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA disponivel:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Ative a GPU: Ambiente de execucao -> Alterar tipo de ambiente de execucao -> GPU'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Restaurar backup da Drive

In [ ]:
def restore(name, dest):
    src = f'{DRIVE_DIR}/backup/{name}/'
    if os.path.isdir(src) and os.listdir(src):
        os.makedirs(dest, exist_ok=True)
        subprocess.run(['rsync', '-a', '--info=progress2', src, dest + '/'], check=True)
        print(f'restaurado: {name} -> {dest}')
    else:
        print(f'nada pra restaurar em {name}')

def backup(name, src):
    dest = f'{DRIVE_DIR}/backup/{name}/'
    os.makedirs(dest, exist_ok=True)
    subprocess.run(['rsync', '-a', '--info=progress2', src + '/', dest], check=True)
    print(f'backup: {src} -> {dest}')

restore('processed', 'data/processed')
restore('cache', 'data/cache')
restore('runs', 'outputs/runs')

## 3. Dataset

In [ ]:
DATASET_ZIP = f'{DRIVE_DIR}/FlickrLogos-32_dataset_v2.zip'
assert os.path.exists(DATASET_ZIP), f'Nao achei {DATASET_ZIP}'

RAW_DIR = f'{PROJECT_DIR}/data/raw/FlickrLogos-32'
expected = f'{RAW_DIR}/dataset_v2/FlickrLogos-v2/classes'

if os.path.isdir(expected):
    print('OK, ja extraido em', RAW_DIR)
else:
    os.makedirs(f'{RAW_DIR}/dataset_v2', exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as zf:
        zf.extractall(f'{RAW_DIR}/dataset_v2')
    assert os.path.isdir(expected)
    print('OK, extraido em', RAW_DIR)

## 4. Treino

In [ ]:
backup_ckpt = f'{DRIVE_DIR}/backup/runs/{RUN_NAME}/model_last.pt'
cmd = ['python', 'scripts/04_train_faster_rcnn.py',
       '--base-config', BASE_CONFIG,
       '--model-config', MODEL_CONFIG,
       '--run-name', RUN_NAME,
       '--checkpoint-backup-dir', f'{DRIVE_DIR}/backup/runs']
if os.path.exists(backup_ckpt):
    cmd += ['--resume-from', backup_ckpt]
    print(f'retomando de {backup_ckpt}')
subprocess.run(cmd, check=True)

In [ ]:
import json

backup('runs', 'outputs/runs')
with open(f'outputs/runs/{RUN_NAME}/run_config.json', encoding='utf-8') as f:
    run_config = json.load(f)
print('conjuntos de treino:', run_config.get('train_splits'))
print('imagens de treino:', run_config.get('train_images'))
print('tempo de treino:', run_config.get('training_time_human'))